# Standard Library => functools

`functools` has tools for working with **functions**: caching results, pre-filling arguments and writing decorators.

| Tool | Purpose | Example |
|---|---|---|
| `@lru_cache(maxsize=128)` | Remember results of past calls | `@lru_cache` |
| `@cache` | Unlimited cache (Python 3.9+) | `@cache` |
| `partial(func, ...)` | Fix some arguments in advance | `partial(int, base=2)` |
| `reduce(func, it, start)` | Combine all items into one value | `reduce(operator.mul, [1, 2, 3])` |
| `@wraps(func)` | Keep the name and docstring of a decorated function | inside decorators |
| `@singledispatch` | Choose an implementation by argument type | |
| `cmp_to_key(func)` | Use an old-style compare function with `sorted` | |
| `cached_property`, `total_ordering` | Class helpers | used inside classes (see Object Oriented Programming) |

```python
from functools import lru_cache, partial, reduce, wraps
```

---

## `lru_cache`: Remember Results

```python
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
```

* A repeated call with the same arguments returns the **saved** result.
* All arguments must be **hashable** (no lists or dicts).
* `fib.cache_info()` shows hits and misses. `fib.cache_clear()` empties the cache.
* Only cache **pure** functions: same arguments always give the same result.

---

## `partial`: Pre-Fill Arguments

```python
binary = partial(int, base=2)
binary("1010")      # 10
```

`partial` creates a new function with some arguments already set.

---

## `reduce`: Fold Into One Value

```python
reduce(lambda a, b: a + b, [1, 2, 3, 4])    # 10
```

* `reduce` returns only the **final** result. `itertools.accumulate` returns every step.
* For sums, products, minimum and maximum, prefer `sum()`, `math.prod()`, `min()` and `max()`.

---

## `wraps`: Well-Behaved Decorators

```python
def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper
```

Without `@wraps`, the decorated function reports the name and docstring of `wrapper`.

---

## `singledispatch`

One function name, different behavior for different argument **types**:

```python
@singledispatch
def describe(value): ...

@describe.register(int)
def _(value): ...
```

---

## `cmp_to_key`

`sorted()` uses a **key function**. `cmp_to_key` adapts an old-style function that compares **two** items and returns a negative, zero or positive number.

## Source

https://docs.python.org/3/library/functools.html

In [ ]:
import operator
from functools import cmp_to_key, lru_cache, partial, reduce, singledispatch, wraps

# lru_cache: remember results
calls = []

@lru_cache(maxsize=None)
def fib(n):
    calls.append(n)
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30), len(calls))                 # each n is computed once
print(fib(30), len(calls))                 # the repeated call is answered from the cache
print(fib.cache_info().hits > 0, fib.cache_info().currsize)
fib.cache_clear()
print(fib.cache_info().currsize)

try:
    lru_cache(maxsize=None)(lambda items: len(items))([1, 2])
except TypeError as error:
    print(type(error).__name__, "for an unhashable argument")

# partial: pre-fill arguments
binary = partial(int, base=2)
print(binary("1010"), binary("11111111"))
power_of_two = partial(pow, 2)
print(power_of_two(10))

# reduce: fold into one value
print(reduce(operator.add, [1, 2, 3, 4]), reduce(operator.mul, [1, 2, 3, 4, 5]))
print(reduce(lambda a, b: a * 10 + b, [1, 2, 3]))            # digits -> number
print(reduce(operator.add, [], 0))                           # a start value handles empty input

# wraps keeps the metadata of the decorated function
def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def plain(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@log_calls
def greet():
    """Say hello."""
    return "hello"

@plain
def greet_plain():
    """Say hello."""
    return "hello"

print(greet.__name__, greet.__doc__)
print(greet_plain.__name__, greet_plain.__doc__)

# singledispatch: choose the implementation by type
@singledispatch
def describe(value):
    return f"something: {value!r}"

@describe.register(int)
def _(value):
    return f"an integer: {value}"

@describe.register(list)
def _(value):
    return f"a list of {len(value)} items"

print(describe(5), "|", describe([1, 2]), "|", describe("text"))

# cmp_to_key: use a compare function with sorted
def by_length_then_alpha(a, b):
    if len(a) != len(b):
        return len(a) - len(b)
    return (a > b) - (a < b)

print(sorted(["pear", "fig", "apple", "kiwi", "date"], key=cmp_to_key(by_length_then_alpha)))